# Nemotron v7 — Train + Submit (All-in-One)

**NVIDIA Nemotron Model Reasoning Challenge**

### What this notebook does
1. **TRAINS** a LoRA adapter on 9,406 verified CoT examples (all 6 categories)
2. **PACKAGES** the adapter into `submission.zip` — ready to submit

### v7 Fixes (v6 scored 0.59 — 3 bugs found)
| Parameter | v6 (0.59) | v7 (fixed) | Why |
|-----------|-----------|------------|-----|
| `gate_proj` in LoRA | Included | **Removed** | Does NOT exist in Nemotron (squared ReLU, no gating) |
| `out_proj` in LoRA | Excluded | **Included** | Both 0.69 and 0.86 notebooks include it |
| `lm_head` in LoRA | Excluded | **Included** | 0.86 notebook trains lm_head — shapes output distribution |
| `num_epochs` | 2 | **3** | 33% more training steps (matches 0.69 config) |
| `lora_alpha` | 128 | **64** | 2:1 ratio (matches 0.69 config that actually worked) |
| `learning_rate` | 1e-5 | **2e-5** | Matches 0.69 config, effective LR = 4e-5 |
| `warmup_steps` | 100 | **50** | Matches 0.69 config |

### Effective LR comparison
- **0.69 config**: lr=2e-5 x alpha/rank=64/32=2.0 → **4e-5** (3 epochs, ~1770 steps)
- **v6 config**: lr=1e-5 x alpha/rank=128/32=4.0 → **4e-5** (2 epochs, ~1180 steps) — same rate but 33% fewer steps!
- **v7 config**: lr=2e-5 x alpha/rank=64/32=2.0 → **4e-5** (3 epochs, ~1770 steps) + `lm_head` + `out_proj`

### Required Kaggle Inputs
- **Model**: `metric/nemotron-3-nano-30b-a3b-bf16`
- **Dataset**: `nemotron-cot-v5` (upload `train_cot_v5_merged.jsonl`)
- **Packages**: `nvidia-nemotron-offline-packages`
- **Accelerator**: GPU (T4/P100/L4/A100)

---
## Part 1: Setup & Dependencies

In [ ]:
# ============================================================
# 1. OFFLINE DEPENDENCY INSTALLATION
# ============================================================
import subprocess, sys, os
from pathlib import Path

def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read().strip()
            rel_pack_path = pth_file.parent / relpath
            if rel_pack_path.exists():
                sys.path.append(str(rel_pack_path))

offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir  = "/kaggle/working/packages"
os.makedirs(target_dir, exist_ok=True)

resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl", "peft"
    ])
    print("Offline packages installed.")

sys.path.append(target_dir)
resolve_python_path(target_dir)

In [ ]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import stat, shutil, zipfile, time, json, re, glob
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# ============================================================
# 3. TRITON / RMSNORM FIXES (critical for NemotronH hybrid arch)
# ============================================================
# NemotronH uses Mamba-2 layers which rely on Triton kernels.
# The default rmsnorm_fn fails on some GPU configs — replace with pure PyTorch.

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

---
## Part 2: Hyperparameters & Data Loading

In [ ]:
# ============================================================
# 4. HYPERPARAMETERS — v7 (matches proven 0.69 config + 0.86 improvements)
# ============================================================
# The 0.69 notebook used: alpha=64, lr=2e-5, 3 epochs, out_proj included
# The 0.86 notebook adds: lm_head
# v6 broke things by: removing out_proj, adding gate_proj (doesn't exist),
#   dropping to 2 epochs

LORA_RANK    = 32
LORA_ALPHA   = 64         # 2:1 ratio (proven in 0.69 notebook)
MAX_SEQ_LEN  = 2048
NUM_EPOCHS   = 3          # 3 epochs (matches 0.69 config)
BATCH_SIZE   = 4          # bump to 8 on A100/L4 if VRAM allows
GRAD_ACCUM   = 4          # effective batch = 16
LR           = 2e-5       # effective LR = lr * alpha/rank = 2e-5 * 2 = 4e-5

MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Data paths — prioritized fallback chain
OUR_DATA_PATHS = [
    "/kaggle/input/nemotron-cot-v5/train_cot_v5_merged.jsonl",        # PRIMARY: 9406 verified examples
    "/kaggle/input/nemotron-dataset/train_cot_v5_merged.jsonl",
    "/kaggle/input/nemotron-cot-v4/train_cot_v4_real.jsonl",          # fallback: 5121 (missing transform)
    "/kaggle/input/nemotron-dataset/train_cot_v4_real.jsonl",
    "/kaggle/input/nemotron-dataset/train_cot_v3_metric_aligned.jsonl",
    "/kaggle/input/nemotron-cot-v3/train_cot_v3_metric_aligned.jsonl",
]
EXTERNAL_CSV_PATHS = [
    "/kaggle/input/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv",
    "/kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv",
]

print(f"Config: {NUM_EPOCHS} epochs, batch={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM} eff, rank {LORA_RANK}, alpha {LORA_ALPHA}, lr {LR}")
print(f"Effective LR = {LR} x {LORA_ALPHA}/{LORA_RANK} = {LR * LORA_ALPHA / LORA_RANK}")

In [ ]:
# ============================================================
# 5. PROGRESS BAR CALLBACK
# ============================================================
class LiveProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar       = None
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(
            total=state.max_steps,
            desc="Training",
            unit="step",
            dynamic_ncols=True,
            file=sys.stdout,
        )
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        elapsed  = time.time() - self.start_time
        step     = state.global_step
        eta      = (elapsed / step) * (state.max_steps - step) if step > 0 else 0
        loss_str = (
            f"loss={state.log_history[-1]['loss']:.4f}"
            if state.log_history and "loss" in state.log_history[-1]
            else "loss=..."
        )
        self.pbar.set_postfix_str(
            f"{loss_str}  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m"
        )
        self.pbar.update(1)
        sys.stdout.flush()

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()

In [ ]:
# ============================================================
# 6. LOAD DATASET — combine BOTH sources for max coverage
# ============================================================
print("Loading datasets...\n")

our_data = []
ext_csv_path = None

# 1) Try our metric-aligned JSONL (prioritized list)
for path in OUR_DATA_PATHS:
    if os.path.exists(path):
        print(f"Found our JSONL: {path}")
        with open(path, 'r') as f:
            for line in f:
                our_data.append(json.loads(line))
        print(f"  -> {len(our_data)} metric-aligned examples")
        break

if not our_data:
    print("Our JSONL not found (searched all candidate paths)")

# 2) Try external CSV as supplemental data
for path in EXTERNAL_CSV_PATHS:
    if os.path.exists(path):
        ext_csv_path = path
        print(f"Found external CSV: {path}")
        break

if not ext_csv_path:
    print("External CSV not found (optional)")

if not our_data and not ext_csv_path:
    raise FileNotFoundError(
        "No training data found!\n"
        "Upload train_cot_v5_merged.jsonl as Kaggle dataset named 'nemotron-cot-v5'\n"
        "OR add kienngx/nemotron-30b-competition-trainingdata-cot-labels as input"
    )

print(f"\nData sources: our_jsonl={len(our_data)}, ext_csv={'YES' if ext_csv_path else 'NO'}")

In [ ]:
# ============================================================
# 7. TOKENIZER & FORMAT — unified pipeline for both sources
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

EVAL_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

all_texts = []

# --- Source A: Our metric-aligned JSONL ---
if our_data:
    print(f"Formatting {len(our_data)} JSONL examples...")
    for example in our_data:
        messages = [m for m in example['messages'] if m['role'] != 'system']

        # Ensure <think> tags wrap reasoning
        assistant_msg = messages[-1]
        if '<think>' not in assistant_msg['content']:
            content = assistant_msg['content']
            boxed_match = re.search(r'(\\boxed\{.*?\})\s*$', content)
            if boxed_match:
                reasoning = content[:boxed_match.start()].strip()
                boxed_answer = boxed_match.group(1)
                messages[-1] = {
                    'role': 'assistant',
                    'content': f"<think>\n{reasoning}\n</think>\n{boxed_answer}"
                }

        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            text = (
                f"<|im_start|>user\n{messages[0]['content']}<|im_end|>\n"
                f"<|im_start|>assistant\n{messages[-1]['content']}<|im_end|>"
            )
        all_texts.append(text)
    print(f"  -> {len(all_texts)} examples from JSONL")

# --- Source B: External CSV (deduplicated) ---
if ext_csv_path:
    import pandas as pd
    ext_df = pd.read_csv(ext_csv_path)
    print(f"\nExternal CSV: {len(ext_df)} rows, columns: {list(ext_df.columns)}")

    # Collect prompts we already have (to avoid duplicates)
    existing_prompts = set()
    if our_data:
        for example in our_data:
            user_content = example['messages'][0]['content'] if example['messages'][0]['role'] == 'user' else ''
            existing_prompts.add(user_content[:200])

    ext_added = 0
    for _, row in ext_df.iterrows():
        prompt = str(row.get('prompt', ''))
        if prompt[:200] in existing_prompts:
            continue

        answer = str(row.get('answer', ''))
        cot = str(row.get('generated_cot', ''))

        if not prompt.strip() or not answer.strip():
            continue

        user_msg = prompt + EVAL_SUFFIX
        assistant_msg = f"<think>\n{cot}\n</think>\n\\boxed{{{answer}}}"

        try:
            messages = [
                {'role': 'user',      'content': user_msg},
                {'role': 'assistant', 'content': assistant_msg},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            text = (
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
            )
        all_texts.append(text)
        ext_added += 1

    print(f"  -> {ext_added} NEW examples from external CSV (after dedup)")

# --- Build final dataset ---
hf_dataset = Dataset.from_dict({'text': all_texts})
print(f"\n{'='*50}")
print(f"TOTAL DATASET: {len(hf_dataset)} examples")
print(f"{'='*50}")
print(f"\nSample (first 400 chars):")
print(hf_dataset[0]['text'][:400])

In [ ]:
# ============================================================
# 8. DROP OVERSIZED SAMPLES (> MAX_SEQ_LEN tokens)
# ============================================================
print(f"Filtering samples > {MAX_SEQ_LEN} tokens...")
before = len(hf_dataset)

def get_token_length(example):
    ids = tokenizer(example['text'], truncation=False,
                    return_attention_mask=False)['input_ids']
    return {'token_len': len(ids)}

hf_dataset = hf_dataset.map(get_token_length, desc="Counting tokens")
hf_dataset = hf_dataset.filter(
    lambda x: x['token_len'] <= MAX_SEQ_LEN,
    desc="Dropping oversized",
)
hf_dataset = hf_dataset.remove_columns(['token_len'])
print(f"  Kept {len(hf_dataset)} / {before}  ({before - len(hf_dataset)} dropped)")

steps_estimate = len(hf_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
print(f"\nEstimated steps : {steps_estimate}")
print(f"Estimated time  : ~{steps_estimate * 8 / 3600:.1f} hrs")

---
## Part 3: Model Loading & LoRA Setup

In [ ]:
# ============================================================
# 9. LOAD MODEL — bf16, NO quantization
# ============================================================
# NemotronH = HYBRID architecture (52 layers):
#   - 23 Attention layers (Transformer)
#   - 23 Mamba-2 layers (SSM)
#   - 6 MoE layers (Mixture of Experts)
# Does NOT support flash_attention_2 — must use "eager" or default

flash_whl = "/kaggle/input/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
if os.path.exists(flash_whl):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-index", flash_whl])
        print("Installed flash_attn wheel (used by internal kernels)")
    except Exception as e:
        print(f"flash_attn install skipped: {e}")

print("Loading base model (bf16, eager attention)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},          # Single GPU
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # Full bf16, NOT quantized
    # NO attn_implementation — NemotronH doesn't support flash_attention_2
)
model.gradient_checkpointing_enable()

# Critical fix: disable fast path for NemotronH
for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False

print(f"Model loaded on GPU")

In [ ]:
# ============================================================
# 10. APPLY FRESH LoRA — matching 0.69 + 0.86 target modules
# ============================================================
# 0.69 notebook targets: q,k,v,o,in,out,up,down (all _proj)
# 0.86 notebook targets: same + lm_head
# v6 BUG: included gate_proj (DOESN'T EXIST in Nemotron — uses squared ReLU)
# v6 BUG: excluded out_proj (both 0.69 and 0.86 include it)

LORA_TARGET_MODULES = [
    # Attention layers (23 layers)
    "q_proj", "k_proj", "v_proj", "o_proj",
    # Mamba-2 SSM layers (23 layers)
    "in_proj", "out_proj",
    # MLP / MoE expert layers
    "up_proj", "down_proj",
    # Output head — from 0.86 notebook (shapes output token distribution)
    "lm_head",
]
# NOTE: gate_proj does NOT exist in Nemotron (it uses squared ReLU, not gated MLP)
# Including non-existent modules can cause silent failures in PEFT

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,          # 64:32 = 2:1 ratio (proven in 0.69)
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.0,               # 0 dropout (matches both 0.69 and 0.86)
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Triton compiler fix
try:
    import triton.backends.nvidia.compiler as nv_compiler
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
except Exception as e:
    print(f"Triton compiler fix skipped: {e}")

---
## Part 4: Training

In [ ]:
# ============================================================
# 11. TRAINING — SFT with speed optimizations
# ============================================================
# Enable tf32 for ~2x matmul speedup on Ampere/Blackwell GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch_fused",       # Fused AdamW: fewer kernel launches
    lr_scheduler_type="cosine",
    warmup_steps=50,                  # Matches 0.69 config
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=True,                    # Pack short sequences = much faster
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=4,        # Parallel data loading
    dataloader_pin_memory=True,      # Faster CPU -> GPU transfer
    dataloader_prefetch_factor=2,    # Prefetch batches
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[LiveProgressCallback()],
)

total_steps = trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else len(hf_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"\nStarting training ({len(hf_dataset)} samples, ~{total_steps} steps)...")
print(f"  batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, eff_batch={BATCH_SIZE*GRAD_ACCUM}")
print(f"  packing=True, fused_adamw=True, tf32=True, workers=4\n")

t0 = time.time()
trainer.train()
elapsed_hrs = (time.time() - t0) / 3600
print(f"\nTraining complete! Time: {elapsed_hrs:.2f} hrs")

---
## Part 5: Save Adapter & Package Submission

In [ ]:
# ============================================================
# 12. SAVE ADAPTER — only adapter files, NO tokenizer (breaks vLLM eval)
# ============================================================
trainer.model.save_pretrained(OUTPUT_DIR)

# Fix base_model_name_or_path to canonical Kaggle model name
config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path) as f:
    adapter_config = json.load(f)

adapter_config["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"

with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)

print(f"base_model_name_or_path -> {adapter_config['base_model_name_or_path']}")

# Verify weights look trained (non-zero norms = good)
try:
    from safetensors import safe_open
    with safe_open(os.path.join(OUTPUT_DIR, "adapter_model.safetensors"),
                   framework="pt") as f:
        keys  = list(f.keys())
        norms = [f.get_tensor(k).norm().item() for k in keys[:5]]
    print(f"\nAdapter tensors: {len(keys)} parameters")
    print(f"First 5 weight norms: {[f'{n:.4f}' for n in norms]}")
    if all(n < 0.001 for n in norms):
        print("WARNING: Norms near 0 — adapter may be untrained. Do NOT submit.")
    else:
        print("Adapter looks healthy (non-zero weights).")
except Exception as e:
    print(f"Could not verify: {e}")

# Show what's in the adapter directory
print(f"\nFiles in {OUTPUT_DIR}:")
for fname in os.listdir(OUTPUT_DIR):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024 / 1024
    print(f"  {fname}  ({size_mb:.2f} MB)")

In [ ]:
# ============================================================
# 13. PACKAGE submission.zip — exactly 2 files
# ============================================================
# The competition expects a zip with:
#   - adapter_config.json
#   - adapter_model.safetensors
# NO tokenizer files, NO other files — just these two.

ZIP_PATH = "/kaggle/working/submission.zip"
REQUIRED = {"adapter_config.json", "adapter_model.safetensors"}

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        if fname in REQUIRED:
            zf.write(os.path.join(OUTPUT_DIR, fname), arcname=fname)

# Verify zip contents
with zipfile.ZipFile(ZIP_PATH) as zf:
    contents = zf.namelist()

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024

print(f"\n{'='*60}")
print(f"  SUBMISSION READY!")
print(f"{'='*60}")
print(f"  File    : {ZIP_PATH}")
print(f"  Size    : {zip_mb:.1f} MB")
print(f"  Contents: {contents}")

assert set(contents) == REQUIRED, f"Wrong files in zip: {contents}"
print(f"\n  Clean 2-file zip — ready to submit!")
print(f"{'='*60}")

---
## Part 6: Adapter Summary & Config Verification

In [ ]:
# ============================================================
# 14. FINAL VERIFICATION — print everything for sanity check
# ============================================================
print("=" * 60)
print("  v7 TRAINING + SUBMISSION SUMMARY")
print("=" * 60)

# Re-read the config from the zip to be sure
with zipfile.ZipFile(ZIP_PATH) as zf:
    with zf.open("adapter_config.json") as f:
        final_cfg = json.load(f)

print(f"\n  Model             : Nemotron-3-Nano-30B-A3B (Hybrid)")
print(f"  base_model_name   : {final_cfg.get('base_model_name_or_path')}")
print(f"  LoRA rank (r)     : {final_cfg.get('r')}")
print(f"  LoRA alpha        : {final_cfg.get('lora_alpha')}")
print(f"  Alpha:Rank ratio  : {final_cfg.get('lora_alpha', 0)}/{final_cfg.get('r', 1)}")
print(f"  Target modules    : {final_cfg.get('target_modules')}")
print(f"  Dropout           : {final_cfg.get('lora_dropout')}")
print(f"  Task type         : {final_cfg.get('task_type')}")
print(f"\n  Dataset           : {len(hf_dataset)} examples (after filtering)")
print(f"  Epochs            : {NUM_EPOCHS}")
print(f"  Learning rate     : {LR}")
print(f"  Effective LR      : {LR * LORA_ALPHA / LORA_RANK}")
print(f"  Effective batch   : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Training time     : {elapsed_hrs:.2f} hrs")
print(f"\n  Submission zip    : {ZIP_PATH} ({zip_mb:.1f} MB)")

# Verify critical settings
targets = final_cfg.get('target_modules', [])
checks = []
checks.append(("base_model = metric/...", final_cfg.get('base_model_name_or_path') == 'metric/nemotron-3-nano-30b-a3b-bf16'))
checks.append(("out_proj IN targets", 'out_proj' in targets))
checks.append(("lm_head IN targets", 'lm_head' in targets))
checks.append(("gate_proj NOT in targets", 'gate_proj' not in targets))
checks.append(("dropout = 0", final_cfg.get('lora_dropout', -1) == 0.0))
checks.append(("rank = 32", final_cfg.get('r') == 32))

print(f"\n  Verification checks:")
all_ok = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_ok = False
    print(f"    [{status}] {name}")

if all_ok:
    print(f"\n  All checks passed! Submit submission.zip to the competition.")
else:
    print(f"\n  WARNING: Some checks failed. Review before submitting.")